In [ ]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio
!pip install datasets transformers
!pip install resampy


In [ ]:
# ============================================================
# CONFIGURAÇÃO — ajuste estas variáveis para o seu ambiente
# ============================================================
import os

# Caminho para a pasta raiz com os arquivos de áudio (.wav) e CSV do dataset.
# No Colab com Google Drive: ex. '/content/drive/MyDrive/asr_data/'
AUDIO_DATA_PATH = os.environ.get("AUDIO_DATA_PATH", "/content/drive/MyDrive/asr_data/")

# Nome do arquivo CSV (colunas obrigatórias: audios, transcription, referencia, audio_classes)
DATASET_CSV_FILENAME = os.environ.get("DATASET_CSV_FILENAME", "dataset.csv")

# Nome para salvar o dataset processado em disco
OUTPUT_DATASET_NAME = os.environ.get("OUTPUT_DATASET_NAME", "asr_dataset")

# Diretório de saída do treinamento (checkpoints e logs)
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "./whisper_finetuned")

# Nome para salvar o modelo treinado
MODEL_OUTPUT_NAME = os.environ.get("MODEL_OUTPUT_NAME", "whisper_model_finetuned")
# ============================================================

In [ ]:
# Configure seu token no Colab Secrets (variável HF_TOKEN) ou faça login abaixo
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

pre = AUDIO_DATA_PATH  # definido na célula de configuração
data = pd.read_csv(AUDIO_DATA_PATH + DATASET_CSV_FILENAME)

In [ ]:
transcript_joined = pd.read_csv(pre + '/transcripts_joined.csv', sep=",")

In [ ]:
# filter transcriptions
print("Transcriptions without filtering: ", transcript_joined.shape[0])
# filter transcriptions
transcript_joined = transcript_joined[transcript_joined['transcription'] != "Sem áudio"]
print("Transcriptions filtering without audio: ", transcript_joined.shape[0])
transcript_joined = transcript_joined[transcript_joined["audio_classes"] != "Inadequado"]
print("Transcriptions filtering without inadequate: ", transcript_joined.shape[0])
transcript_joined = transcript_joined[(transcript_joined.transcription.str.len() <= transcript_joined.transcription.str.len().median())]
print("Transcriptions median filtering: ", transcript_joined.shape[0])
transcript_joined = transcript_joined[(transcript_joined.transcription.str.len() >=20)]
print("Transcriptions min filtering: ", transcript_joined.shape[0])
print("Transcriptions median filtering: ", transcript_joined.shape[0])
transcript_joined = transcript_joined[transcript_joined['transcription'] != "Sem fala"]
print("Transcriptions filtering without sem fala: ", transcript_joined.shape[0])





In [ ]:
import glob
audios = glob.glob(pre + "*.wav")
audios = [file.split('/')[-1] for file in audios]

In [ ]:
from pydub import AudioSegment
from scipy.io import wavfile
import resampy
import numpy as np

def read_wav_file(file_path, target_sample_rate=16000):
    sample_rate, data = wavfile.read(file_path)

    # Verifica se o áudio é estéreo e converte para mono, se necessário
    if len(data.shape) > 1:
        data = np.mean(data, axis=1)

    # Reamostragem do áudio
    if sample_rate != target_sample_rate:
        if len(data) >= 2:
            data = resampy.resample(data, sample_rate, target_sample_rate)
        else:
            return None

    return target_sample_rate, data

In [ ]:
all_data = []
back_path = AUDIO_DATA_PATH  # definido na célula de configuração
n_transcripts_to_eval = len(transcript_joined)
for i, row in transcript_joined.iloc[0:n_transcripts_to_eval].iterrows():
    print(i)
    audio = read_wav_file(back_path + row['audios'])
    if audio is not None:
        data = {
            'audio': {
                'path': back_path + row['audios'],
                'array': audio[1],
                'sampling_rate': 16000
            },
            'sentence': row['referencia']
        }
        all_data.append(data)

In [ ]:
df = pd.DataFrame([{
    'audio_path': item['audio']['path'],
    'audio_array': item['audio']['array'],
    'sampling_rate': item['audio']['sampling_rate'],
    'sentence': item['sentence']
} for item in all_data])

In [ ]:
from datasets import Dataset, DatasetDict

dataset = Dataset.from_pandas(df[['audio_path', 'audio_array', 'sampling_rate', 'sentence']])

In [ ]:
dataset.save_to_disk(AUDIO_DATA_PATH + OUTPUT_DATASET_NAME)

In [ ]:
import json

# Create a Dataset object
dataset = Dataset.from_pandas(df[['audio_path', 'audio_array', 'sampling_rate', 'sentence']])

# Split the dataset into train, validation, and test sets
dataset_dict = dataset.train_test_split(test_size=0.1)
train_val = dataset_dict['train'].train_test_split(test_size=0.1)

# Create a DatasetDict
asr_dataset = DatasetDict({
    'train': train_val['train'],
    'validation': train_val['test'],
    'test': dataset_dict['test']
})

In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")


In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-medium", language="portuguese", task="transcribe", skip_special_tokens=True)


In [ ]:
input_str = asr_dataset["train"][0]["sentence"]
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=True)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")


In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-medium", language="portuguese", task="transcribe")


In [ ]:
print(asr_dataset["train"][0])


In [ ]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio_array"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio, sampling_rate=16000).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"], max_length=448, truncation=True).input_ids
    return batch


In [ ]:
asr_dataset = asr_dataset.map(prepare_dataset, remove_columns=asr_dataset.column_names["train"])


In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")


In [ ]:
model.generation_config.language = "portuguese"
model.generation_config.task = "transcribe"


In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


In [ ]:
import evaluate

metric = evaluate.load("wer")


In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=48,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=5000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=100,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=asr_dataset["train"],
    eval_dataset=asr_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


In [ ]:
trainer.train()


In [ ]:
model.save_pretrained(AUDIO_DATA_PATH + MODEL_OUTPUT_NAME)

In [ ]:
metric.

In [ ]:
eval_results = trainer.evaluate()

In [ ]:
eval_results

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs